# Project Answer — live demo on Colab

Serves the **whole demo site** — presentation, the frozen-run replay, **and** live
inference — from a single Colab GPU runtime:

- **Ollama** serving `gemma4:e4b` (LLM) + `embeddinggemma:300m` (embedder)
- **The Answer pipeline** (LangGraph, Chroma, MiniLM cross-encoder reranker)
- **FastAPI (`api.py`)** serving the built React site at `/` and a streaming
  `POST /ask` SSE endpoint
- Exposed to the world via a free **Cloudflare quick tunnel**

After the setup cells finish (~8–12 min on first run), the tunnel cell prints a
public `https://*.trycloudflare.com` URL. Open it in any browser — the site is
served from this runtime, so **live mode works out of the box** (same origin), and
the saved-run replay works too.

## Before you run anything

**GPU runtime.** Runtime → Change runtime type → T4 GPU (free tier is enough).

That's it — no accounts, no tokens, no signups.

## How it stays alive

Colab idle-kills the runtime after ~90 min of inactivity. The last cell prints a
heartbeat every 5 min — leave that tab open while the demo URL is being used.

## 1. Install Ollama, cloudflared, and system deps

In [ ]:
%%capture
!sudo apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

## 2. Start Ollama (localhost)

No tunnel needed for Ollama itself — the API server runs on the same Colab runtime
and talks to it via `localhost:11434`.

In [ ]:
import os, subprocess, threading, time

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"

def _serve_ollama():
    subprocess.Popen(
        ["ollama", "serve"],
        env={**os.environ},
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

threading.Thread(target=_serve_ollama, daemon=True).start()
time.sleep(5)

import requests
print(requests.get("http://127.0.0.1:11434/").text)

## 3. Pull the models

Both models are cached after the first pull — re-running this cell in the same
session is fast. First pull is ~3–5 min.

In [ ]:
!ollama pull gemma4:e4b
!ollama pull embeddinggemma:300m
!ollama list

## 4. Clone the repo + install Python deps

`uv sync` pulls torch, sentence-transformers, chromadb, fastapi, etc. — ~5 min on a
fresh runtime.

In [ ]:
import os
os.chdir("/content")

REPO_URL = "https://github.com/w-sliman/answer.git"
PROJECT_DIR = "/content/answer-project"

if not os.path.isdir(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)

# uv = the project's Python package manager. Pin to ~/.local/bin so the rest of
# the notebook finds it via the PATH update below.
!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"

!uv sync

## 5. Install Playwright browsers for crawl4ai

The fetch node uses crawl4ai, which drives Playwright. One-time install per runtime.

In [ ]:
!uv run crawl4ai-setup

## 6. Build the web frontend

Compiles the React site to `web/dist`, which `api.py` serves at `/`. Colab ships
Node.js; the guard installs it only if it's missing. ~1–2 min.

In [ ]:
# Node 20 (only if Colab's image lacks it)
!node --version 2>/dev/null || (curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - >/dev/null 2>&1 && sudo apt-get install -y nodejs >/dev/null 2>&1)
!node --version && npm --version

!cd /content/answer-project/web && npm install --no-audit --no-fund && npm run build
!ls /content/answer-project/web/dist

## 7. Write `.env`

Ollama lives on the same machine, so `OLLAMA_BASE_URL` is just `localhost`. The chat
model is `gemma4:e4b`.

In [ ]:
env_lines = [
    "OLLAMA_BASE_URL=http://localhost:11434",
    "OLLAMA_MODEL=gemma4:e4b",
    "OLLAMA_EMBED_MODEL=embeddinggemma:300m",
    "OLLAMA_NUM_CTX=32768",
]

with open("/content/answer-project/.env", "w") as f:
    f.write("\n".join(env_lines) + "\n")

!cat /content/answer-project/.env

## 8. Warm up the models

Pre-loads both models into GPU memory so the first live question doesn't pay the
~30s cold-load cost. We match the pipeline's `num_ctx` (32768) so the KV cache is
sized correctly the first time it's allocated.

In [ ]:
import requests, time

t0 = time.time()
requests.post(
    "http://127.0.0.1:11434/api/generate",
    json={"model": "gemma4:e4b", "prompt": "hi", "stream": False,
          "options": {"num_ctx": 32768}},
    timeout=600,
).raise_for_status()
print(f"✓ gemma4:e4b loaded ({time.time() - t0:.0f}s)")

t1 = time.time()
requests.post(
    "http://127.0.0.1:11434/api/embed",
    json={"model": "embeddinggemma:300m", "input": "warmup"},
    timeout=600,
).raise_for_status()
print(f"✓ embeddinggemma:300m loaded ({time.time() - t1:.0f}s)")

## 9. Start the API server (`api.py`)

FastAPI serves the built site at `/`, a health check at `/health`, and the streaming
`POST /ask` endpoint. Bound to all interfaces so the tunnel can reach it; we wait for
`/health` before continuing.

In [ ]:
import subprocess, time, requests, os

LOG_PATH = "/content/api.log"

api_proc = subprocess.Popen(
    ["uv", "run", "uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/answer-project",
    stdout=open(LOG_PATH, "w"),
    stderr=subprocess.STDOUT,
    env={**os.environ},
)

for i in range(180):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200 and r.json().get("ok"):
            print(f"✓ API up after {i}s")
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("✗ API didn't come up in 180s — log tail:")
    !tail -40 {LOG_PATH}

## 10. Expose the API via a Cloudflare quick tunnel

No account, no token. `cloudflared` opens an outbound connection to Cloudflare's edge,
which routes a public URL back to port 8000. The URL is ephemeral — it changes per
Colab session and lives only while the tunnel process runs. Because the site is served
from the same origin, its live mode works with no extra configuration.

In [ ]:
import subprocess, re, time, os

LOG_PATH = "/content/cloudflared.log"

tunnel_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=open(LOG_PATH, "w"),
    stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
for i in range(60):
    time.sleep(1)
    if not os.path.exists(LOG_PATH):
        continue
    with open(LOG_PATH) as f:
        text = f.read()
    m = re.search(r"https://(?:[a-z0-9]+-){2,}[a-z0-9]+\.trycloudflare\.com", text)
    if m:
        PUBLIC_URL = m.group(0)
        break

if PUBLIC_URL:
    print("=" * 64)
    print("ANSWER — live demo, public URL:")
    print("  " + PUBLIC_URL)
    print("=" * 64)
    print("Open in any browser. The full site loads here, and live mode is enabled.")
    print("Keep this Colab tab open while in use. The URL changes every session.")
else:
    print("✗ cloudflared didn't print a URL in 60s — log tail:")
    !tail -40 {LOG_PATH}

## 11. Keep the runtime alive

Colab idle-kills the runtime after ~90 min of no activity. This cell prints a
heartbeat every 5 min so the runtime stays counted as 'active.' Leave it running while
the demo is being used; stop it (■) when you're done.

Logs, if something looks off: the API log is at `/content/api.log` and the tunnel log
is at `/content/cloudflared.log` — `!tail -50 <path>` in a new cell to inspect.

In [ ]:
import time, datetime

while True:
    print(f"[alive] {datetime.datetime.now().isoformat(timespec='seconds')}", flush=True)
    time.sleep(300)